# Norris Phase 5 — evaluation, paired tests, over-editing and failure analysis

Run after Phase 4 is complete (`phase4_summary.json` status `phase4_complete`). A100 runtime. ≈20–30 min (COMET on ~25k unique triplets, 48 bootstrap tests, flags).

In [ ]:
# 1 — Mount Drive, clone the Norris repo at REF, load Colab Secrets (never printed).
from pathlib import Path
import os, subprocess, sys, json, hashlib, zipfile
from google.colab import drive, userdata
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/norris')
REPO = Path('/content/csudh-norris-ape-2026')
REF = 'phase4-main'
URL = 'https://github.com/SHAZAN01/csudh-norris-ape-2026.git'
import shutil
shutil.rmtree(REPO, ignore_errors=True)  # always a fresh clone: robust to stale folders and rewritten branches
subprocess.run(['git', 'clone', '--branch', REF, URL, str(REPO)], check=True)
COMMIT = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
print('REPO COMMIT:', COMMIT)
assert (REPO / 'src/common/norris_protocol.py').exists(), 'Wrong branch content: push the patched branch first.'
for name in ['HF_TOKEN', 'OPENAI_API_KEY']:
    try:
        os.environ[name] = userdata.get(name) or ''
    except Exception:
        os.environ[name] = ''
    print(name, 'available' if os.environ[name] else 'MISSING (enable it in Colab Secrets if this notebook needs it)')
LOGS = ROOT / 'outputs/logs/colab'
LOGS.mkdir(parents=True, exist_ok=True)

def sh(cmd, log='run.log'):
    cmd = [str(c) for c in cmd]
    print('$', ' '.join(cmd)[:240], flush=True)
    env = os.environ.copy()
    for k in ['PYTHONPATH', 'UV_SYSTEM_PYTHON', 'VIRTUAL_ENV']:
        env.pop(k, None)
    env['TOKENIZERS_PARALLELISM'] = 'false'
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env) as p, \
            open(LOGS / log, 'a', encoding='utf-8') as f:
        for line in p.stdout:
            print(line, end='', flush=True)
            f.write(line)
    if p.returncode:
        raise RuntimeError(f'Stopped (exit {p.returncode}). Copy the error above and send it; log: {LOGS / log}')

sh([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
def venv(path, py):
    if not Path(path, 'bin/python').exists():
        sh([sys.executable, '-m', 'uv', 'venv', '--python', py, path])
    return Path(path, 'bin/python')
def uv(py, *pkgs):
    sh([sys.executable, '-m', 'uv', 'pip', 'install', '--python', py, *pkgs], log='install.log')
print('SETUP OK')

In [ ]:
# 2 — Freeze and Phase 4 status
FREEZE = ROOT / 'outputs/frozen/protocol_freeze.json'
F = json.loads(FREEZE.read_text())
bad = [f for f, h in F['code_sha256'].items() if hashlib.sha256((REPO / f).read_bytes()).hexdigest() != h]
assert F['freeze_declared'] and not bad, bad
assert json.loads((ROOT / 'outputs/phase4/phase4_summary.json').read_text())['status'] == 'phase4_complete'
print('OK')

In [ ]:
# 3 — Evaluation environment

# Evaluation environment (same COMET stack as the audited pilot evaluation)
EVAL_PY = venv('/content/env_eval', '3.11')
uv(EVAL_PY, 'torch==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cu124')
uv(EVAL_PY, 'torch==2.6.0', 'unbabel-comet==2.2.7', 'transformers==4.49.0', 'huggingface-hub==0.36.0', 'numpy==1.26.4',
   'sacrebleu==2.6.0', 'torchmetrics==0.10.3', 'setuptools==80.9.0', 'langdetect==1.0.9', 'pydantic==2.12.3',
   'rapidfuzz==3.10.1', 'pysbd==0.3.4', 'matplotlib==3.9.2')
sh([EVAL_PY, '-c', "import torch, comet, transformers, numpy, sacrebleu, langdetect, pydantic; "
    "print('torch', torch.__version__, 'transformers', transformers.__version__, 'numpy', numpy.__version__); assert torch.cuda.is_available()"])

In [ ]:
# 4 — Full Phase 5 evaluation + LaTeX tables + paper macros
sh([EVAL_PY, REPO / 'tests/phase5/test_stats.py'])
sh([EVAL_PY, REPO / 'src/evaluation/evaluate_main_experiments.py', '--root', ROOT, '--freeze', FREEZE, '--repo', REPO],
   log='phase5_eval.log')
EV = Path(json.loads((ROOT / 'outputs/phase5/latest.json').read_text())['output_dir'])
os.environ['MPLBACKEND'] = 'Agg'  # Colab's inline backend cannot load in the separate venv
sh([EVAL_PY, REPO / 'src/analysis/generate_main_figures.py', '--eval-dir', EV, '--out', EV / 'figures'])

In [ ]:
# 5 — Evidence ZIP for Report 5 and the manuscript (numeric only: no benchmark text)
from google.colab import files
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
zp = ROOT / f'outputs/reports/Norris_Phase5_Evidence_{stamp}.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in EV.rglob('*'):
        if p.is_file() and 'comet_cache' not in p.parts:
            z.write(p, str(p.relative_to(EV)))
    for p in LOGS.glob('phase5*.log'):
        z.write(p, 'logs/' + p.name)
print(zp, round(zp.stat().st_size / 1e6, 1), 'MB'); files.download(str(zp))

In [ ]:
# 6 — Source-review cases (PRIVATE: contains FLORES+ text and model outputs). Rule-based selection fixed in code:
#     per condition, largest strong-draft over-edit, largest improvement, first digit-loss and first language-drift flag.
EV = Path(json.loads((ROOT / 'outputs/phase5/latest.json').read_text())['output_dir'])
sh([EVAL_PY, REPO / 'src/analysis/extract_review_cases.py', '--root', ROOT, '--eval-dir', EV], log='phase5_cases.log')
from google.colab import files
files.download(str(EV / 'review_cases_PRIVATE.json'))